# PRiM-score: nonlinear Gaussian example

This notebook generates one nonlinear additive Gaussian SEM, with **500 normal samples**, **10 anomalous samples**, and two true root causes. It then runs **Algorithm 1 (PRiM-Score-Standard)**, **Algorithm 2 (PRiM-Score-Local)**, and **PRiM-Score-NOTEARS** using exact Gaussian-process conditional mechanisms.

## 1. Imports and experimental settings

Run the notebook from the repository root after `python -m pip install -e .`.

In [1]:
import time

import numpy as np
import pandas as pd

from prim_score import (
    learn_cpdag,
    learn_notears_dag_isolated,
    local_prim,
    prim_notears,
    standard_prim,
)

SEM_TYPE = 'nonlinear'
N_NORMAL = 500
N_ANOMALOUS = 10
DELTA = 1.5
TRUE_ROOTS = ['X0', 'X4']
TOP_K = len(TRUE_ROOTS)
print(f'SEM={SEM_TYPE}, n={N_NORMAL}, m={N_ANOMALOUS}, delta={DELTA}, true roots={TRUE_ROOTS}')

SEM=nonlinear, n=500, m=10, delta=1.5, true roots=['X0', 'X4']


## 2. Generate one nonlinear Gaussian dataset

The structural functions are nonlinear (`sin` and `tanh`) and every structural noise term is Gaussian. Anomalous samples are generated by adding a mean shift of `DELTA` to the noises of `X0` and `X4`.

In [2]:
def simulate_nonlinear_gaussian(n, seed, interventions=None):
    interventions = interventions or {}
    rng = np.random.default_rng(seed)
    noise = rng.normal(loc=0.0, scale=1.0, size=(n, 8))
    for node, shift in interventions.items():
        noise[:, node] += shift

    x = np.zeros_like(noise)
    x[:, 0] = noise[:, 0]
    x[:, 1] = noise[:, 1]
    x[:, 4] = noise[:, 4]
    x[:, 6] = noise[:, 6]
    x[:, 2] = 1.2 * np.sin(x[:, 0]) + 0.8 * np.tanh(x[:, 1]) + noise[:, 2]
    x[:, 3] = 0.9 * np.sin(x[:, 2]) + 1.0 * np.tanh(x[:, 4]) + noise[:, 3]
    x[:, 5] = 0.9 * np.sin(x[:, 3]) + 0.7 * np.tanh(x[:, 6]) + noise[:, 5]
    x[:, 7] = 0.9 * np.tanh(x[:, 5]) + noise[:, 7]
    return pd.DataFrame(x, columns=[f'X{i}' for i in range(8)])

normal_data = simulate_nonlinear_gaussian(N_NORMAL, seed=42)
anomalous_data = simulate_nonlinear_gaussian(
    N_ANOMALOUS, seed=2026, interventions={0: DELTA, 4: DELTA}
)

print('Normal data shape:   ', normal_data.shape)
print('Anomalous data shape:', anomalous_data.shape)
print('True root causes:    ', TRUE_ROOTS)
normal_data.head()

Normal data shape:    (500, 8)
Anomalous data shape: (10, 8)
True root causes:     ['X0', 'X4']


,X0,X1,X2,X3,X4,X5,X6,X7
0,0.304717,-1.039984,0.488174,0.402277,-1.951035,-0.860812,0.127840,-0.943251
1,-0.016801,-0.853044,0.305112,1.114087,0.066031,2.240328,0.467509,0.020549
2,0.368751,-0.958883,0.715968,0.358003,-0.184862,0.222716,1.222541,0.042665
3,-0.428328,-0.352134,-0.236724,0.545167,0.412733,1.578473,2.141648,0.420112
4,-0.512243,-0.813773,-0.509514,0.576538,-0.113947,-0.823795,-0.824481,0.041176


## 3. Estimate the CPDAG with PC+RCIT

The default `learn_cpdag(normal_data)` call uses **PC+RCIT**, matching this nonlinear data-generating process. Only the 500 normal samples are used for structure learning. The true graph is displayed solely for comparison and is not passed to either PRiM algorithm. For a separate linear experiment, explicitly pass `model_type='linear'` to use XGES.

In [3]:
true_adjacency = np.zeros((8, 8), dtype=int)
for parent, child in [(0, 2), (1, 2), (2, 3), (4, 3), (3, 5), (6, 5), (5, 7)]:
    true_adjacency[parent, child] = 1

true_cpdag = pd.DataFrame(
    true_adjacency, index=normal_data.columns, columns=normal_data.columns
)
discovery = learn_cpdag(normal_data, random_state=0)
cpdag = discovery['cpdag']
print(f"Selected learner: {discovery['learner']}")
print(f"Resolved model type: {discovery['model_type']}")
print(f"Structure-learning time: {discovery['runtime_sec']:.3f} s")
print(f"CI-test alpha: {discovery.get('alpha', 'not used by XGES')}")
pd.concat({'True CPDAG (reference only)': true_cpdag, 'Estimated CPDAG': cpdag})

Selected learner: PC+RCIT
Resolved model type: nonlinear
Structure-learning time: 2.125 s
CI-test alpha: 0.05


X0  X1  X2  X3  X4  X5  X6  X7
True CPDAG (reference only) X0   0   0   1   0   0   0   0   0
                            X1   0   0   1   0   0   0   0   0
                            X2   0   0   0   1   0   0   0   0
                            X3   0   0   0   0   0   1   0   0
                            X4   0   0   0   1   0   0   0   0
                            X5   0   0   0   0   0   0   0   1
                            X6   0   0   0   0   0   1   0   0
                            X7   0   0   0   0   0   0   0   0
Estimated CPDAG             X0   0   0   1   0   0   0   0   0
                            X1   0   0   1   0   0   0   0   0
                            X2   0   0   0   1   0   0   0   0
                            X3   0   0   0   0   0   0   0   0
                            X4   0   0   0   1   0   0   0   0
                            X5   0   0   0   1   0   0   1   1
                            X6   0   0   0   0   0   1   0   0
                            X7   0   0   0   0   0   1   0   0

## 4. Run Algorithm 1, Algorithm 2, and PRiM-NOTEARS

Algorithms 1 and 2 receive the PC+RCIT **estimated CPDAG**. PRiM-NOTEARS first learns its own nonlinear NOTEARS-MLP DAG from the same 500 normal samples. All three methods then use the default exact Gaussian-process residual mechanism, so no GP options need to be passed.

In [5]:
start = time.perf_counter()
algorithm_1 = standard_prim(
    normal_data,
    anomalous_data,
    cpdag,
    top_k=TOP_K,
)
algorithm_1_time = time.perf_counter() - start

start = time.perf_counter()
algorithm_2 = local_prim(
    normal_data,
    anomalous_data,
    cpdag,
    top_k=TOP_K,
)
algorithm_2_time = time.perf_counter() - start

start = time.perf_counter()
notears_dag = learn_notears_dag_isolated(normal_data)
notears_graph_time = time.perf_counter() - start

start = time.perf_counter()
prim_notears_result = prim_notears(
    normal_data,
    anomalous_data,
    notears_dag,
    top_k=TOP_K,
)
prim_notears_score_time = time.perf_counter() - start
print(f'Nonlinear NOTEARS DAG learned in {notears_graph_time:.3f} s')
notears_dag

Nonlinear NOTEARS DAG learned in 5.509 s


,X0,X1,X2,X3,X4,X5,X6,X7
X0,0,0,1,0,0,0,0,0
X1,0,0,1,0,0,0,0,0
X2,0,0,0,1,0,0,0,0
X3,0,0,0,0,0,1,0,1
X4,0,0,0,1,0,0,0,0
X5,0,0,0,0,0,0,0,0
X6,0,0,0,0,0,1,0,1
X7,0,0,0,0,0,1,0,0


## 5. Display the true and estimated root causes

In [6]:
def recall_at_k(predicted, truth):
    return len(set(predicted) & set(truth)) / len(truth)

summary = pd.DataFrame(
    [
        {
            'Method': 'Algorithm 1: PRiM-Score-Standard',
            'Graph learner': discovery['learner'],
            'True roots': TRUE_ROOTS,
            'Predicted roots': algorithm_1['predicted_roots'],
            f'Recall@{TOP_K}': recall_at_k(algorithm_1['predicted_roots'], TRUE_ROOTS),
            'Runtime (s)': algorithm_1_time,
        },
        {
            'Method': 'Algorithm 2: PRiM-Score-Local',
            'Graph learner': discovery['learner'],
            'True roots': TRUE_ROOTS,
            'Predicted roots': algorithm_2['predicted_roots'],
            f'Recall@{TOP_K}': recall_at_k(algorithm_2['predicted_roots'], TRUE_ROOTS),
            'Runtime (s)': algorithm_2_time,
        },
        {
            'Method': 'PRiM-Score-NOTEARS',
            'Graph learner': 'NOTEARS-MLP',
            'True roots': TRUE_ROOTS,
            'Predicted roots': prim_notears_result['predicted_roots'],
            f'Recall@{TOP_K}': recall_at_k(
                prim_notears_result['predicted_roots'], TRUE_ROOTS
            ),
            'Runtime (s)': notears_graph_time + prim_notears_score_time,
        },
    ]
)
summary

,Method,Graph learner,True roots,Predicted roots,Recall@2,Runtime (s)
0,Algorithm 1: PRiM-Score-Standard,PC+RCIT,"[X0, X4]","[X0, X4]",1.0,0.221818
1,Algorithm 2: PRiM-Score-Local,PC+RCIT,"[X0, X4]","[X0, X4]",1.0,0.087473
2,PRiM-Score-NOTEARS,NOTEARS-MLP,"[X0, X4]","[X0, X4]",1.0,5.577846


In [7]:
score_table = pd.DataFrame(
    {
        'Algorithm 1 score': algorithm_1['scores'],
        'Algorithm 2 score': algorithm_2['scores'],
        'PRiM-NOTEARS score': prim_notears_result['scores'],
    }
).sort_values('Algorithm 1 score', ascending=False)
score_table

,Algorithm 1 score,Algorithm 2 score,PRiM-NOTEARS score
X0,1.976202,1.976202,1.976202
X4,1.111222,1.111222,1.111222
X1,0.723373,0.723373,0.723373
X5,0.454701,0.382606,0.453108
X3,0.260632,0.260632,0.415397
X6,0.239016,0.029885,0.029885
X7,0.030840,0.030840,0.038746
X2,0.019621,0.019621,0.019621
